# 03 · Portfolio VaR

Risk on a small book. Two paths:
1. `nq.risk.symbol_var` — pass tickers + a date window; the backend pulls
   verified returns from the Golden Record itself.
2. `nq.risk.portfolio_var` — supply your own return series per symbol.

In [ ]:
from nyquist import Nyquist

nq = Nyquist()

## Path 1 — symbol VaR (returns fetched server-side)

In [ ]:
book = nq.risk.symbol_var(
    symbols=["SPY", "QQQ", "TLT"],
    start_date="2026-01-01T00:00:00",
    end_date="2026-04-01T00:00:00",
    weights=[0.5, 0.3, 0.2],
    confidence=0.99,
)
book

## Path 2 — historical portfolio VaR from your own returns

Here we derive daily returns locally from `nq.marketdata.history`, then
send them to the portfolio-VaR engine.

In [ ]:
tickers = ["SPY", "QQQ"]
weights = [0.6, 0.4]
returns_data = {}
for t in tickers:
    hist = nq.marketdata.history(t, days=365)
    if not hist.empty:
        returns_data[t] = hist["close"].pct_change().dropna().tolist()

if all(returns_data.get(t) for t in tickers):
    result = nq.risk.portfolio_var(
        tickers, weights, returns_data,
        confidence_level=0.99, portfolio_value=1_000_000,
    )
else:
    result = {"note": "no Golden Record history in this environment"}
result

## Single-series VaR

`nq.risk.var` takes a raw return series and one method (`parametric` / `historical` / `monte_carlo`).

In [ ]:
rets = returns_data.get("SPY", [])
if rets:
    var = nq.risk.var(rets, method="historical", confidence=0.95)
else:
    var = {"note": "no returns available"}
var